# RAIS 2022

Livro municipal de emprego formal para o ano de referência 2022.
As definições das variáveis estão em **dictionary.csv**.
A tabela salva inclui a chave do município, o ano e os nomes de origem.


## Importações


In [7]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
from unidecode import unidecode


## Configuração

In [8]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"

RAW_RAIS_DIR = BASE_DIR / "raw" / "rais"
TRUSTED_RAIS_DIR = BASE_DIR / "trusted" / "rais"
TRUSTED_IBGE_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_RAIS_DIR, TRUSTED_RAIS_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Diretório

In [9]:
path_rais = RAW_RAIS_DIR / "rais_2022.csv"


## Nome de Colunas

In [10]:
def normalizar_nome_coluna(col: str) -> str:
    """
    Normaliza nome de coluna:
    - remove acentos;
    - deixa minúsculo;
    - troca caracteres especiais por underscore.
    """
    col = str(col).strip()
    col = unidecode(col)
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df


def parse_numero(s: pd.Series) -> pd.Series:
    """
    Interpreta textos numéricos: decimal com ponto (por exemplo 1234.56) ou padrão brasileiro (1.234,56).
    """
    s_str = (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA,
            "-": pd.NA,
            "...": pd.NA,
        })
    )
    normalized = s_str.copy()
    br_mask = normalized.str.contains(",", na=False)
    normalized.loc[br_mask] = (
        normalized.loc[br_mask]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(normalized, errors="coerce")


## RAIS 2022

In [11]:
def ler_arquivo_rais_municipio(path_arquivo: Path) -> pd.DataFrame:
    """
    Lê arquivo RAIS agregado por município.

    Suporta:
    - .csv
    - .csv.gz
    - .parquet
    - .xlsx
    - .xls
    """
    path_arquivo = Path(path_arquivo)

    if not path_arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path_arquivo}")

    suffixes = "".join(path_arquivo.suffixes).lower()

    if suffixes.endswith(".parquet"):
        df = pd.read_parquet(path_arquivo)

    elif suffixes.endswith(".csv.gz"):
        df = pd.read_csv(path_arquivo, dtype="string", compression="gzip")

    elif suffixes.endswith(".csv"):
        try:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="utf-8")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="utf-8")

        except UnicodeDecodeError:
            df = pd.read_csv(path_arquivo, dtype="string", encoding="latin1")

            if df.shape[1] == 1:
                df = pd.read_csv(path_arquivo, sep=";", dtype="string", encoding="latin1")

    elif suffixes.endswith(".xlsx") or suffixes.endswith(".xls"):
        df = pd.read_excel(path_arquivo, dtype="string")

    else:
        raise ValueError(f"Extensão não suportada: {suffixes}")

    df = normalizar_colunas(df)

    return df


In [12]:
df_rais_raw = ler_arquivo_rais_municipio(path_rais)

print(df_rais_raw.shape)
df_rais_raw.head()


(5571, 11)


,ano,sigla_uf,id_municipio,nome_municipio,qtd_vinculos_rais,qtd_vinculos_ativos_3112,remuneracao_media,massa_remuneracao_media,qtd_vinculos_ativos_comercio_servicos,qtd_vinculos_ativos_industria_construcao,qtd_vinculos_ativos_agro
0,2022,AC,1200013,Acrelândia,1456,1006,2358.23909340659,3433596.1200000029,746,144,116
1,2022,AC,1200054,Assis Brasil,737,550,2853.7556445047471,2103217.9099999964,527,5,18
2,2022,AC,1200104,Brasiléia,4244,2931,2207.3583388312959,9368028.7899999842,2168,721,42
3,2022,AC,1200138,Bujari,2882,1904,1944.5579458709246,5604216.0000000214,1591,32,281
4,2022,AC,1200179,Capixaba,1024,779,2664.4352734374993,2728381.719999996,615,39,125


In [13]:
df_rais_raw.dtypes


ano                                         string[python]
sigla_uf                                    string[python]
id_municipio                                string[python]
nome_municipio                              string[python]
qtd_vinculos_rais                           string[python]
qtd_vinculos_ativos_3112                    string[python]
remuneracao_media                           string[python]
massa_remuneracao_media                     string[python]
qtd_vinculos_ativos_comercio_servicos       string[python]
qtd_vinculos_ativos_industria_construcao    string[python]
qtd_vinculos_ativos_agro                    string[python]
dtype: object

## Padronização

In [14]:
def padronizar_rais_2022_municipio(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza a RAIS 2022 agregada por município.

    Saída:
        codigo_ibge
        ano
        variáveis agregadas da RAIS
    """
    df = normalizar_colunas(df_raw)

    colunas_obrigatorias = [
        "ano",
        "sigla_uf",
        "id_municipio",
        "nome_municipio",
        "qtd_vinculos_rais",
        "qtd_vinculos_ativos_3112",
        "remuneracao_media",
        "massa_remuneracao_media",
        "qtd_vinculos_ativos_comercio_servicos",
        "qtd_vinculos_ativos_industria_construcao",
        "qtd_vinculos_ativos_agro",
    ]

    ausentes = [col for col in colunas_obrigatorias if col not in df.columns]

    if ausentes:
        raise ValueError(
            f"As seguintes colunas obrigatórias não foram encontradas: {ausentes}. "
            f"Colunas disponíveis: {df.columns.tolist()}"
        )

    out = pd.DataFrame()

    out["codigo_ibge"] = (
        df["id_municipio"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")

    out["sigla_uf_origem"] = df["sigla_uf"].astype("string").str.upper().str.strip()
    out["nome_municipio_origem"] = df["nome_municipio"].astype("string").str.strip()

    cols_numericas = [
        "qtd_vinculos_rais",
        "qtd_vinculos_ativos_3112",
        "remuneracao_media",
        "massa_remuneracao_media",
        "qtd_vinculos_ativos_comercio_servicos",
        "qtd_vinculos_ativos_industria_construcao",
        "qtd_vinculos_ativos_agro",
    ]

    for col in cols_numericas:
        out[col] = parse_numero(df[col])

    # Garante ano 2022
    out = out[out["ano"] == 2022].copy()

    # Remove duplicidades, caso existam
    duplicadas = out.duplicated(["codigo_ibge", "ano"]).sum()

    if duplicadas > 0:
        print(f"Atenção: existem {duplicadas} duplicidades por codigo_ibge-ano. Consolidando.")

        out = (
            out
            .groupby(["codigo_ibge", "ano"], as_index=False)
            .agg(
                sigla_uf_origem=("sigla_uf_origem", "first"),
                nome_municipio_origem=("nome_municipio_origem", "first"),
                qtd_vinculos_rais=("qtd_vinculos_rais", "sum"),
                qtd_vinculos_ativos_3112=("qtd_vinculos_ativos_3112", "sum"),
                remuneracao_media=("remuneracao_media", "mean"),
                massa_remuneracao_media=("massa_remuneracao_media", "sum"),
                qtd_vinculos_ativos_comercio_servicos=("qtd_vinculos_ativos_comercio_servicos", "sum"),
                qtd_vinculos_ativos_industria_construcao=("qtd_vinculos_ativos_industria_construcao", "sum"),
                qtd_vinculos_ativos_agro=("qtd_vinculos_ativos_agro", "sum"),
            )
        )

    # Shares setoriais sobre vínculos ativos
    denom = out["qtd_vinculos_ativos_3112"].replace(0, np.nan)

    out["share_vinculos_comercio_servicos"] = (
        out["qtd_vinculos_ativos_comercio_servicos"] / denom
    )

    out["share_vinculos_industria_construcao"] = (
        out["qtd_vinculos_ativos_industria_construcao"] / denom
    )

    out["share_vinculos_agro"] = (
        out["qtd_vinculos_ativos_agro"] / denom
    )
    out = (
        out
        .sort_values(["codigo_ibge", "ano"])
        .reset_index(drop=True)
    )

    return out


In [15]:
rais_2022_municipio = padronizar_rais_2022_municipio(df_rais_raw)

print(rais_2022_municipio.shape)
rais_2022_municipio.head()


(5571, 14)


,codigo_ibge,ano,sigla_uf_origem,nome_municipio_origem,qtd_vinculos_rais,qtd_vinculos_ativos_3112,remuneracao_media,massa_remuneracao_media,qtd_vinculos_ativos_comercio_servicos,qtd_vinculos_ativos_industria_construcao,qtd_vinculos_ativos_agro,share_vinculos_comercio_servicos,share_vinculos_industria_construcao,share_vinculos_agro
0,1100015,2022,RO,Alta Floresta D'Oeste,4564,3251,2466.129347,11255414.34,2414,403,434,0.742541,0.123962,0.133497
1,1100023,2022,RO,Ariquemes,33381,21346,2474.878798,82613929.150001,16512,3564,1270,0.773541,0.166963,0.059496
2,1100031,2022,RO,Cabixi,1103,888,2266.4365,2499879.46,551,109,228,0.620495,0.122748,0.256757
3,1100049,2022,RO,Cacoal,31878,19905,2199.601631,70118900.79,15657,3655,593,0.786586,0.183622,0.029792
4,1100056,2022,RO,Cerejeiras,4390,2935,2714.083478,11914826.47,2320,272,343,0.79046,0.092675,0.116865


In [16]:
cols_rais = [
    "codigo_ibge",
    "ano",
    "sigla_uf_origem",
    "nome_municipio_origem",
    "qtd_vinculos_rais",
    "qtd_vinculos_ativos_3112",
    "remuneracao_media",
    "massa_remuneracao_media",
    "qtd_vinculos_ativos_comercio_servicos",
    "qtd_vinculos_ativos_industria_construcao",
    "qtd_vinculos_ativos_agro",
    "share_vinculos_comercio_servicos",
    "share_vinculos_industria_construcao",
    "share_vinculos_agro",
]


In [17]:
rais_2022_municipio[cols_rais]


,codigo_ibge,ano,sigla_uf_origem,nome_municipio_origem,qtd_vinculos_rais,qtd_vinculos_ativos_3112,remuneracao_media,massa_remuneracao_media,qtd_vinculos_ativos_comercio_servicos,qtd_vinculos_ativos_industria_construcao,qtd_vinculos_ativos_agro,share_vinculos_comercio_servicos,share_vinculos_industria_construcao,share_vinculos_agro
0,1100015,2022,RO,Alta Floresta D'Oeste,4564,3251,2466.129347,11255414.34,2414,403,434,0.742541,0.123962,0.133497
1,1100023,2022,RO,Ariquemes,33381,21346,2474.878798,82613929.150001,16512,3564,1270,0.773541,0.166963,0.059496
2,1100031,2022,RO,Cabixi,1103,888,2266.4365,2499879.46,551,109,228,0.620495,0.122748,0.256757
3,1100049,2022,RO,Cacoal,31878,19905,2199.601631,70118900.79,15657,3655,593,0.786586,0.183622,0.029792
4,1100056,2022,RO,Cerejeiras,4390,2935,2714.083478,11914826.47,2320,272,343,0.79046,0.092675,0.116865
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5566,5222054,2022,GO,Vicentinópolis,4619,2644,2471.360593,11415214.58,1071,1054,519,0.405068,0.398638,0.196293
5567,5222203,2022,GO,Vila Boa,3287,1698,1609.571308,5290660.89,530,80,1088,0.312132,0.047114,0.640754
5568,5222302,2022,GO,Vila Propício,2309,1621,2413.826657,5573525.75,633,357,631,0.3905,0.220234,0.389266
5569,5300108,2022,DF,Brasília,1760152,1250053,5860.501196,10315372901.098618,1126433,117118,6502,0.901108,0.09369,0.005201


In [18]:
rais_2022_municipio[cols_rais].to_parquet(
    TRUSTED_IBGE_DIR / "rais_2022_municipio.parquet",
    index=False
)
